# DCA drone-detection — Ground-truth generator (Kaggle)

This notebook generates **only the ground truth**: the Layer-1 scenario JSONs and the Layer-2 `.npz` + `.meta.json` tracks (positions, velocities, headings, and per-sensor physics labels). **No rendering** — that comes later in Blender/acoular.

Each code cell below writes one module to disk with `%%writefile`, then the last cells run the generator and sanity-check the output. **Run top to bottom (Run All).**

To edit sensor locations, change `SENSOR_POSITIONS` in the `config.py` cell.

## 1. Modules

In [ ]:
%%writefile geometry.py
"""
Coordinate system and spatial boundaries.

CONVENTION (locked):
    - 1 unit = 1 meter, everywhere (Blender, acoular, RF all assume SI/meters).
    - Right-handed ENU:  X = East,  Y = North,  Z = Up.
    - Origin = a FIXED ground landmark (e.g. base of the control tower), z = 0 at ground.
      Record its real lat/lon so the local frame can be tied back to reality later.

The "Region" is the global volume objects are allowed to occupy. You said you'd
trace the tightest polygon around the perimeter in Blender and pad it 100-200 m.
That polygon (list of (x, y) tuples) + an altitude band IS the region. Replace the
placeholder REGION_POLYGON below with the coordinates you read off the Blender model.
"""

from __future__ import annotations
import numpy as np


class Region:
    """A horizontal polygon footprint with an altitude band [z_min, z_max]."""

    def __init__(self, polygon_xy, z_min, z_max):
        self.poly = np.asarray(polygon_xy, dtype=float)   # (K, 2)
        assert self.poly.shape[0] >= 3, "need >=3 polygon vertices"
        self.z_min = float(z_min)
        self.z_max = float(z_max)
        # precompute edge lengths for perimeter sampling
        nxt = np.roll(self.poly, -1, axis=0)
        self._edge_len = np.linalg.norm(nxt - self.poly, axis=1)
        self._cum = np.cumsum(self._edge_len)
        self._perim = float(self._cum[-1])

    @property
    def centroid(self):
        return self.poly.mean(axis=0)

    @property
    def bbox(self):
        return (self.poly[:, 0].min(), self.poly[:, 0].max(),
                self.poly[:, 1].min(), self.poly[:, 1].max())

    def contains_xy(self, x, y) -> bool:
        """Ray-casting point-in-polygon test."""
        poly = self.poly
        inside = False
        n = len(poly)
        j = n - 1
        for i in range(n):
            xi, yi = poly[i]
            xj, yj = poly[j]
            if ((yi > y) != (yj > y)) and \
               (x < (xj - xi) * (y - yi) / (yj - yi + 1e-12) + xi):
                inside = not inside
            j = i
        return inside

    def contains(self, p) -> bool:
        return self.contains_xy(p[0], p[1]) and (self.z_min <= p[2] <= self.z_max)

    def random_point(self, rng, z_min=None, z_max=None):
        """Uniform-ish random point inside the polygon at a random altitude."""
        x0, x1, y0, y1 = self.bbox
        for _ in range(200):
            x = rng.uniform(x0, x1)
            y = rng.uniform(y0, y1)
            if self.contains_xy(x, y):
                break
        zlo = self.z_min if z_min is None else z_min
        zhi = self.z_max if z_max is None else z_max
        z = rng.uniform(zlo, zhi)
        return np.array([x, y, z], dtype=float)

    def random_perimeter_point(self, rng, z=None):
        """A point on the polygon boundary (good 'entry/exit' spawn points)."""
        s = rng.uniform(0.0, self._perim)
        e = int(np.searchsorted(self._cum, s))
        e = min(e, len(self.poly) - 1)
        seg_start = 0.0 if e == 0 else self._cum[e - 1]
        frac = (s - seg_start) / max(self._edge_len[e], 1e-9)
        a = self.poly[e]
        b = self.poly[(e + 1) % len(self.poly)]
        xy = a + frac * (b - a)
        if z is None:
            z = 0.0
        return np.array([xy[0], xy[1], z], dtype=float)


# ----------------------------------------------------------------------------
# small numpy-only helpers (no scipy dependency for the core pipeline)
# ----------------------------------------------------------------------------
def gaussian_kernel(sigma: float):
    radius = max(1, int(round(3 * sigma)))
    x = np.arange(-radius, radius + 1)
    k = np.exp(-(x ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def smooth_columns(a: np.ndarray, sigma: float) -> np.ndarray:
    """Edge-padded Gaussian smoothing of each column of (N, D)."""
    if sigma <= 0 or len(a) < 3:
        return a
    k = gaussian_kernel(sigma)
    pad = len(k) // 2
    out = np.empty_like(a)
    for j in range(a.shape[1]):
        col = np.pad(a[:, j], pad, mode="edge")
        out[:, j] = np.convolve(col, k, mode="valid")[: a.shape[0]]
    return out


def ou_jitter(n: int, dt: float, sigma: float, tau: float, rng) -> np.ndarray:
    """Ornstein-Uhlenbeck (mean-reverting) noise: a *smooth* random walk that
    wanders but stays bounded. This is the right tool for organic wobble -
    plain Brownian motion would drift away."""
    x = np.zeros((n, 3))
    theta = 1.0 / max(tau, 1e-3)
    coef = sigma * np.sqrt(2 * theta * dt)
    for i in range(1, n):
        x[i] = x[i - 1] - theta * x[i - 1] * dt + coef * rng.standard_normal(3)
    return x


In [ ]:
%%writefile config.py
"""
Global configuration. This is the file you edit to match the real DCA model.
Everything downstream reads from here.
"""

from __future__ import annotations
import numpy as np
from geometry import Region

# ---------------------------------------------------------------------------
# CLOCKS  (one master timeline; each renderer subsamples it at its own rate)
# ---------------------------------------------------------------------------
MASTER_FS = 100.0       # Hz  -> the ground-truth track is sampled at 100 Hz
AUDIO_FS = 44100        # Hz  -> acoular renders at full audio rate
AUDIO_FS_MODEL = 4000   # Hz  -> what your DroneCNN expects (resample down)
VIDEO_FPS = 24          # frames/s for the visual renderer
RF_SNAPSHOT_LEN = 2048  # samples per IQ snapshot fed to IQCNN
RF_FS = 1_000_000       # Hz  baseband sample rate for an RF snapshot

SPEED_OF_SOUND = 343.0  # m/s
NOISE_FLOOR_DBM = -90.0 # receiver thermal noise floor (kTB + NF), for RF SNR labels

# Audio level calibration (RELATIVE levels; tune against real recordings for
# sim-to-real). acoular received amplitude ~ source_rms / distance, so we map a
# profile's reference SPL (dB at 1 m) to a source rms relative to a 78 dB anchor.
AUDIO_REF_DB = 78.0     # anchor: a small quad at 1 m
AUDIO_REF_RMS = 30.0    # source rms assigned to the anchor
AUDIO_BG_RMS = 0.02     # per-channel background noise rms

# ---------------------------------------------------------------------------
# REGION  (REPLACE with the polygon you trace in Blender + 100-200 m margin)
# Placeholder ~ a 2.2 km x 3.4 km box around the origin, DCA-ish footprint.
# ---------------------------------------------------------------------------
REGION_POLYGON = [
    (-1100.0, -1700.0),
    (1100.0, -1700.0),
    (1100.0, 1700.0),
    (-1100.0, 1700.0),
]
ALT_MIN, ALT_MAX = 2.0, 150.0      # drones realistically live 0-120 m (~400 ft)
REGION = Region(REGION_POLYGON, ALT_MIN, ALT_MAX)


# ---------------------------------------------------------------------------
# SENSORS  (placeholder: 23 nodes on the perimeter, facing inward.
#           REPLACE positions with your surveyed sensor locations.)
#   - each acoustic sensor = 1 microphone  -> 1 audio channel
#   - each RF sensor        = 1 receiver   -> 1 IQ stream
#     (a real direction-finder is a mic/antenna *array*; for v1 one element per
#      node is enough, and TDOA across the 23 nodes already gives localization)
#   - cameras are expensive, so only a subset carry one.
#   - fov_deg is the full horizontal field of view; the renderer turns it into a
#     Blender camera angle. pitch_deg tilts the boresight up (drones fly high).
# ---------------------------------------------------------------------------
def _make_perimeter_sensors(region: Region, n=23, inset=60.0, cam_every=3):
    x0, x1, y0, y1 = region.bbox
    x0 += inset; x1 -= inset; y0 += inset; y1 -= inset
    corners = np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]])
    seg = np.roll(corners, -1, axis=0) - corners
    seglen = np.linalg.norm(seg, axis=1)
    perim = seglen.sum()
    cum = np.cumsum(seglen)
    cen = np.array([(x0 + x1) / 2, (y0 + y1) / 2])
    sensors = []
    for i in range(n):
        s = perim * i / n
        e = int(np.searchsorted(cum, s)); e = min(e, 3)
        start = 0.0 if e == 0 else cum[e - 1]
        frac = (s - start) / max(seglen[e], 1e-9)
        xy = corners[e] + frac * seg[e]
        yaw = float(np.degrees(np.arctan2(cen[1] - xy[1], cen[0] - xy[0])))
        modal = ["mic", "rf"] + (["cam"] if i % cam_every == 0 else [])
        sensors.append({
            "id": f"s{i:02d}",
            "pos": [float(xy[0]), float(xy[1]), 4.0],   # ~4 m mast height
            "yaw_deg": yaw,
            "pitch_deg": 8.0,
            "fov_deg": 70.0,
            "modalities": modal,
        })
    return sensors

# ---------------------------------------------------------------------------
# SENSOR PLACEMENT
# Real nodes are at SPECIFIC surveyed locations, not on a box. Fill
# SENSOR_POSITIONS with your nodes; each row:
#   (id, x, y, z, yaw_deg, pitch_deg, fov_deg, [modalities])
#     x, y, z : metres from the origin; z = HEIGHT ABOVE GROUND (mast/roof height)
#     yaw_deg : boresight heading, measured from +X (East) toward +Y (North)
#     pitch_deg : boresight tilt up from horizontal (0 = level)
#     fov_deg : full horizontal field of view (cameras); ignored for mic/rf
#     modalities : subset of ["mic","rf","cam"]; default ["mic","rf"]
# Leave SENSOR_POSITIONS = None to auto-place on the perimeter so the pipeline
# runs before you have real coordinates.
# ---------------------------------------------------------------------------
SENSOR_POSITIONS = None
# Example once you have Blender coords:
# SENSOR_POSITIONS = [
#     ("s00",  312.0, -145.0, 4.0, 210, 8, 70, ["mic", "rf", "cam"]),
#     ("s01", -260.0,  410.0, 6.0,  45, 6, 70, ["mic", "rf"]),
#     ...
# ]


def _sensors_from_positions(rows):
    out = []
    for r in rows:
        sid, x, y, z, yaw, pitch, fov = r[:7]
        modal = list(r[7]) if len(r) > 7 else ["mic", "rf"]
        out.append({"id": sid, "pos": [float(x), float(y), float(z)],
                    "yaw_deg": float(yaw), "pitch_deg": float(pitch),
                    "fov_deg": float(fov), "modalities": modal})
    return out


SENSORS = (_sensors_from_positions(SENSOR_POSITIONS) if SENSOR_POSITIONS
           else _make_perimeter_sensors(REGION))


# ---------------------------------------------------------------------------
# OBJECT PROFILES
# Drones differ mostly in size/speed/RF; one shape of motion, scaled by a
# per-instance speed multiplier. Add real models (DJI etc.) as more entries.
# ---------------------------------------------------------------------------
DRONE_PROFILES = {
    "small_quad": dict(
        mesh="small_quad.glb", rpm=[15000, 14950, 13500, 13050], blades=2,
        size_m=0.35, v_cruise=12.0, v_max=18.0, climb_rate=4.0,
        ref_spl_db=78.0,                         # SPL at 1 m, drives audio labels
        rf_carrier_hz=2.44e9, rf_tx_dbm=20.0, rf_mod="ofdm",
    ),
    "large_quad": dict(
        mesh="large_quad.glb", rpm=[7000, 6980, 6500, 6450], blades=2,
        size_m=0.9, v_cruise=16.0, v_max=23.0, climb_rate=6.0,
        ref_spl_db=86.0,
        rf_carrier_hz=5.8e9, rf_tx_dbm=23.0, rf_mod="ofdm",
    ),
}

BIRD_PROFILES = {
    "gull": dict(
        mesh="gull.glb", size_m=0.4, v_cruise=10.0, v_max=16.0,
        ref_spl_db=70.0, source_audio="bird_gull.wav",   # use a real recording
        emits_rf=False,
    ),
}

PLANE_PROFILES = {
    "regional_jet": dict(
        mesh="regional_jet.glb", size_m=30.0, v_taxi=8.0, v_takeoff=80.0,
        ref_spl_db=110.0, source_audio="jet_flyby.wav",  # reuse your Airport Noise set
        emits_rf=False,
    ),
}

# Planes don't wander - they follow a small set of named routes (you refine
# these from the runway/taxiway geometry). Runway 1/19 at DCA is ~2185 m, N-S,
# so approach/departure run mostly along +/- Y. Pauses get inserted for variation.
# Each path = ordered list of (x, y, z) waypoints in airport coordinates.
PLANE_PATHS = {
    "land_19": [(0, 1600, 120), (0, 900, 60), (0, 200, 6), (0, -800, 4), (0, -1090, 4)],
    "takeoff_01": [(0, -1090, 4), (0, -400, 4), (0, 300, 25), (0, 1000, 90), (0, 1600, 140)],
    "taxi_a":     [(0, -1090, 4), (120, -1000, 4), (240, -700, 4), (240, -300, 4)],
}


In [ ]:
%%writefile schema.py
"""
The two-layer contract.

LAYER 1  -  Scenario  (authoring / "g-code")          -> stored as JSON
    Sparse, human-readable, version-controllable, randomizable. A list of
    objects, each with a small vocabulary of maneuvers, plus sensors + env.
    acoular has no notion of "orbit"; maneuvers are an authoring convenience
    that the compiler expands into waypoints.

LAYER 2  -  Track  (ground truth / renderer input)    -> stored as .npz + .meta.json
    Dense per-sample arrays. THIS is what every renderer streams. It is NOT
    JSON: per-sample positions/velocities/labels for many objects x 23 sensors
    are numeric arrays, and JSON would be huge and slow. So:
        track_XXXX.npz        -> all arrays (compressed)
        track_XXXX.meta.json  -> the scalars (ids, classes, is_threat) so the
                                 dataset stays greppable.

The compiler (compile_track.py) turns Layer 1 -> Layer 2.
Renderers (render_audio / render_visual / render_rf) consume Layer 2 ONLY.
"""

from __future__ import annotations
from dataclasses import dataclass, field, asdict
from typing import Any
import json
import numpy as np


# ============================== LAYER 1 ====================================

@dataclass
class Maneuver:
    type: str                       # spawn|goto|orbit|climb|descend|hover|waypoints
    params: dict[str, Any] = field(default_factory=dict)


@dataclass
class ObjectSpec:
    id: str
    cls: str                        # "drone" | "bird" | "plane"
    profile: str                    # key into *_PROFILES
    start_delay: float = 0.0        # seconds after t=0 before this object begins
    speed_mult: float = 1.0         # per-instance speed scaling
    maneuvers: list[Maneuver] = field(default_factory=list)


@dataclass
class SensorSpec:
    id: str
    pos: list[float]
    yaw_deg: float
    pitch_deg: float
    fov_deg: float
    modalities: list[str]


@dataclass
class EnvironmentSpec:
    c: float = 343.0
    wind_mps: list[float] = field(default_factory=lambda: [0.0, 0.0, 0.0])
    weather: str = "clear"
    bg_noise: str = "airport_ambient.wav"


@dataclass
class Scenario:
    scenario_id: str
    seed: int
    duration_s: float
    frame: dict
    region_polygon: list[list[float]]
    alt_band: list[float]
    environment: EnvironmentSpec
    objects: list[ObjectSpec]
    sensors: list[SensorSpec]

    def to_json(self, path):
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)

    @staticmethod
    def from_json(path) -> "Scenario":
        with open(path) as f:
            d = json.load(f)
        d["environment"] = EnvironmentSpec(**d["environment"])
        d["objects"] = [
            ObjectSpec(
                id=o["id"], cls=o["cls"], profile=o["profile"],
                start_delay=o["start_delay"], speed_mult=o["speed_mult"],
                maneuvers=[Maneuver(**m) for m in o["maneuvers"]],
            ) for o in d["objects"]
        ]
        d["sensors"] = [SensorSpec(**s) for s in d["sensors"]]
        return Scenario(**d)


# ============================== LAYER 2 ====================================
# Stored flat in an .npz. Keys:
#   t                                  (N,)        master timeline, seconds
#   threat_present                     (N,) bool   any drone active at t
#   obj/<id>/active                    (N,) bool
#   obj/<id>/pos                       (N,3)       NaN where inactive
#   obj/<id>/vel                       (N,3)
#   obj/<id>/yaw_deg, /pitch_deg       (N,)
#   lbl/<id>/<sensorid>/range_m        (N,)
#   lbl/<id>/<sensorid>/radial_mps     (N,)   +ve = receding
#   lbl/<id>/<sensorid>/az_deg,/el_deg (N,)   relative to sensor boresight
#   lbl/<id>/<sensorid>/in_fov         (N,) bool (geometric cone; occlusion is
#                                                 added later by the visual pass)
#   lbl/<id>/<sensorid>/spl_db         (N,)   expected received SPL  (audio)
#   lbl/<id>/<sensorid>/rx_dbm         (N,)   expected received power (RF)

class Track:
    def __init__(self, arrays: dict[str, np.ndarray], meta: dict):
        self.a = arrays
        self.meta = meta

    # convenience accessors -------------------------------------------------
    @property
    def t(self):            return self.a["t"]
    @property
    def fs(self):           return self.meta["fs"]
    @property
    def object_ids(self):   return [o["id"] for o in self.meta["objects"]]
    def cls_of(self, oid):  return next(o["cls"] for o in self.meta["objects"] if o["id"] == oid)
    def profile_of(self, oid): return next(o["profile"] for o in self.meta["objects"] if o["id"] == oid)
    def pos(self, oid):     return self.a[f"obj/{oid}/pos"]
    def vel(self, oid):     return self.a[f"obj/{oid}/vel"]
    def active(self, oid):  return self.a[f"obj/{oid}/active"]
    def label(self, oid, sid, fieldname): return self.a[f"lbl/{oid}/{sid}/{fieldname}"]

    def save(self, npz_path, meta_path):
        np.savez_compressed(npz_path, **self.a)
        with open(meta_path, "w") as f:
            json.dump(self.meta, f, indent=2)

    @staticmethod
    def load(npz_path, meta_path) -> "Track":
        npz = np.load(npz_path)
        arrays = {k: npz[k] for k in npz.files}
        with open(meta_path) as f:
            meta = json.load(f)
        return Track(arrays, meta)


In [ ]:
%%writefile maneuvers.py
"""
Maneuver vocabulary. Each maneuver expands into a (M,3) array of positions
sampled at MASTER_FS, continuous with the previous maneuver's end point.

Vocabulary (intentionally small - this is all you need):
    spawn      {pos:[x,y,z]}                          set start position (first maneuver)
    goto       {to:[x,y,z], speed}                    straight line at constant speed
    climb      {to_alt, speed}                        change altitude (xy held)
    descend    {to_alt, speed}                        (same as climb; sign differs)
    hover      {duration, jitter?}                    stay put (optional OU wobble)
    orbit      {center:[x,y,z], radius, turns|duration, speed}   circular loiter
    waypoints  {points:[[x,y,z],...], speed}          free polyline (escape hatch)

Turns/curves don't need a dedicated primitive: orbit covers loiter circles, and
the compiler's Gaussian smoothing rounds the joints between goto segments.
"""

from __future__ import annotations
import numpy as np
from geometry import ou_jitter


def _n_samples(duration, fs):
    return max(2, int(round(duration * fs)))


def _line(p0, p1, speed, fs):
    p0 = np.asarray(p0, float); p1 = np.asarray(p1, float)
    dist = np.linalg.norm(p1 - p0)
    dur = dist / max(speed, 1e-3)
    n = _n_samples(dur, fs)
    s = np.linspace(0, 1, n)[:, None]
    return p0[None, :] + s * (p1 - p0)[None, :]


def expand(maneuvers, fs, speed_mult, rng):
    """Return (positions (M,3), duration_s). Drops duplicated join samples."""
    segments = []
    cur = np.zeros(3)
    for m in maneuvers:
        t = m.type
        p = m.params
        if t == "spawn":
            cur = np.asarray(p["pos"], float)
            segments.append(cur[None, :])
        elif t == "goto":
            target = np.asarray(p["to"], float)
            seg = _line(cur, target, p["speed"] * speed_mult, fs)
            segments.append(seg); cur = seg[-1]
        elif t in ("climb", "descend"):
            target = cur.copy(); target[2] = p["to_alt"]
            seg = _line(cur, target, p["speed"] * speed_mult, fs)
            segments.append(seg); cur = seg[-1]
        elif t == "hover":
            n = _n_samples(p["duration"], fs)
            seg = np.repeat(cur[None, :], n, axis=0)
            if p.get("jitter", 0.0) > 0:
                seg = seg + ou_jitter(n, 1.0 / fs, sigma=p["jitter"], tau=1.5, rng=rng)
            segments.append(seg); cur = seg[-1]
        elif t == "orbit":
            center = np.asarray(p["center"], float)
            R = float(p["radius"])
            v = p["speed"] * speed_mult
            ang_speed = v / max(R, 1e-3)                    # rad/s
            a0 = np.arctan2(cur[1] - center[1], cur[0] - center[0])
            if "turns" in p:
                total = 2 * np.pi * p["turns"]
            else:
                total = ang_speed * p["duration"]
            direction = p.get("dir", 1.0)
            dur = total / max(ang_speed, 1e-6)
            n = _n_samples(dur, fs)
            ang = a0 + direction * np.linspace(0, total, n)
            seg = np.stack([center[0] + R * np.cos(ang),
                            center[1] + R * np.sin(ang),
                            np.full(n, center[2])], axis=1)
            segments.append(seg); cur = seg[-1]
        elif t == "waypoints":
            pts = [np.asarray(q, float) for q in p["points"]]
            for q in pts:
                seg = _line(cur, q, p["speed"] * speed_mult, fs)
                segments.append(seg); cur = seg[-1]
        else:
            raise ValueError(f"unknown maneuver type: {t}")

    # concatenate, dropping the duplicated first sample of each later segment
    out = [segments[0]]
    for seg in segments[1:]:
        out.append(seg[1:] if len(seg) > 1 else seg)
    pos = np.concatenate(out, axis=0)
    duration = (len(pos) - 1) / fs
    return pos, duration


In [ ]:
%%writefile generator.py
"""
Random scenario generation.

REALISTIC RANDOM WALKS - the principle:
    A pure random walk looks drunk. Real motion is *piecewise-structured*: a
    short sequence of deliberate maneuvers with randomized PARAMETERS, then
    smoothed at the joints. So we randomize positions/speeds/radii/durations
    within physical limits, not the per-sample direction. For organic wobble
    we overlay a small Ornstein-Uhlenbeck process (mean-reverting; see
    geometry.ou_jitter) during hovers - it drifts but stays bounded.

    drone : enter -> climb -> 1-3 waypoints -> optional loiter/orbit -> hover
            -> exit. Structured, smooth.
    bird  : erratic short hops with big heading changes at low altitude, with
            occasional perch (descend to ~ground + pause) and takeoff. Heading
            follows velocity (not locked north).
    plane : NOT free - picks one of a few named routes (PLANE_PATHS) and inserts
            random holds/pauses + a speed pick for variation.

VARIATION FOR CHEAP:  every object has a start_delay, and the generator can be
    called many times with the same object set but permuted delays/orderings.
    Because everything is placed on one master clock by start_delay, shifting
    delays yields genuinely different multi-object scenes at ~zero cost.
"""

from __future__ import annotations
import numpy as np
from schema import (Scenario, ObjectSpec, SensorSpec, EnvironmentSpec, Maneuver)
import config as C


def _drone_mission(region, prof, speed_mult, rng):
    v = prof["v_cruise"]
    entry = region.random_perimeter_point(rng, z=rng.uniform(2, 8))
    cruise_alt = rng.uniform(30, min(120, region.z_max))
    mans = [Maneuver("spawn", {"pos": entry.tolist()}),
            Maneuver("climb", {"to_alt": cruise_alt, "speed": prof["climb_rate"]})]
    n_wp = rng.integers(1, 4)
    last = entry.copy(); last[2] = cruise_alt
    for _ in range(int(n_wp)):
        wp = region.random_point(rng, z_min=cruise_alt - 15, z_max=cruise_alt + 15)
        mans.append(Maneuver("goto", {"to": wp.tolist(), "speed": float(rng.uniform(0.6, 1.0) * v)}))
        last = wp
    if rng.random() < 0.6:                       # loiter over a point of interest
        center = region.random_point(rng, z_min=cruise_alt, z_max=cruise_alt)
        R = float(rng.uniform(40, 120))
        # approach the circle first so the orbit doesn't jump radially
        start = center.copy(); start[0] += R
        mans.append(Maneuver("goto", {"to": start.tolist(), "speed": float(0.7 * v)}))
        mans.append(Maneuver("orbit", {"center": center.tolist(), "radius": R,
                                       "turns": float(rng.uniform(0.75, 2.5)),
                                       "speed": float(rng.uniform(0.5, 0.9) * v),
                                       "dir": float(rng.choice([-1, 1]))}))
        last = start
    if rng.random() < 0.7:
        mans.append(Maneuver("hover", {"duration": float(rng.uniform(3, 10)), "jitter": 0.4}))
    exit_pt = region.random_perimeter_point(rng, z=rng.uniform(5, 30))
    mans.append(Maneuver("goto", {"to": exit_pt.tolist(), "speed": float(rng.uniform(0.8, 1.0) * prof["v_max"])}))
    return mans


def _bird_mission(region, prof, speed_mult, rng):
    v = prof["v_cruise"]
    cur = region.random_point(rng, z_min=5, z_max=60)
    mans = [Maneuver("spawn", {"pos": cur.tolist()})]
    n_hops = rng.integers(4, 9)
    for _ in range(int(n_hops)):
        if rng.random() < 0.2:                   # perch: drop to ground + pause
            ground = cur.copy(); ground[2] = 0.5
            mans.append(Maneuver("descend", {"to_alt": 0.5, "speed": float(rng.uniform(2, 5))}))
            mans.append(Maneuver("hover", {"duration": float(rng.uniform(1, 4)), "jitter": 0.05}))
            mans.append(Maneuver("climb", {"to_alt": float(rng.uniform(10, 50)), "speed": float(rng.uniform(2, 6))}))
            cur = ground
        else:                                    # short erratic hop
            hop = region.random_point(rng, z_min=5, z_max=60)
            # keep hops short-ish for sporadic feel
            d = hop - cur
            if np.linalg.norm(d[:2]) > 250:
                hop[:2] = cur[:2] + d[:2] / np.linalg.norm(d[:2]) * rng.uniform(60, 250)
            mans.append(Maneuver("goto", {"to": hop.tolist(), "speed": float(rng.uniform(0.5, 1.0) * prof["v_max"])}))
            cur = hop
    return mans


def _plane_mission(prof, speed_mult, rng):
    name = rng.choice(list(C.PLANE_PATHS.keys()))
    pts = [list(map(float, q)) for q in C.PLANE_PATHS[name]]
    is_taxi = name.startswith("taxi")
    spd = prof["v_taxi"] if is_taxi else float(rng.uniform(0.6, 1.0) * prof["v_takeoff"])
    mans = [Maneuver("spawn", {"pos": pts[0]})]
    for i, q in enumerate(pts[1:], start=1):
        seg_speed = prof["v_taxi"] if (is_taxi or q[2] < 10) else spd
        mans.append(Maneuver("goto", {"to": q, "speed": seg_speed}))
        if rng.random() < 0.25:                  # random hold for variation
            mans.append(Maneuver("hover", {"duration": float(rng.uniform(2, 8))}))
    return mans, name


def random_scenario(seed, duration_s=45.0, n_drones=1, n_birds=1, n_planes=1):
    rng = np.random.default_rng(seed)
    region = C.REGION
    objects = []
    k = 0

    for _ in range(n_drones):
        pname = rng.choice(list(C.DRONE_PROFILES.keys()))
        prof = C.DRONE_PROFILES[pname]
        sm = float(rng.uniform(0.85, 1.15))
        objects.append(ObjectSpec(
            id=f"drone{k}", cls="drone", profile=pname,
            start_delay=float(rng.uniform(0, duration_s * 0.5)), speed_mult=sm,
            maneuvers=_drone_mission(region, prof, sm, rng)))
        k += 1
    for _ in range(n_birds):
        pname = rng.choice(list(C.BIRD_PROFILES.keys()))
        prof = C.BIRD_PROFILES[pname]
        objects.append(ObjectSpec(
            id=f"bird{k}", cls="bird", profile=pname,
            start_delay=float(rng.uniform(0, duration_s * 0.6)), speed_mult=1.0,
            maneuvers=_bird_mission(region, prof, 1.0, rng)))
        k += 1
    for _ in range(n_planes):
        pname = rng.choice(list(C.PLANE_PROFILES.keys()))
        prof = C.PLANE_PROFILES[pname]
        mans, _route = _plane_mission(prof, 1.0, rng)
        objects.append(ObjectSpec(
            id=f"plane{k}", cls="plane", profile=pname,
            start_delay=float(rng.uniform(0, duration_s * 0.4)), speed_mult=1.0,
            maneuvers=mans))
        k += 1

    wind_dir = rng.uniform(0, 2 * np.pi)
    wind_spd = float(rng.uniform(0, 7))
    env = EnvironmentSpec(
        c=C.SPEED_OF_SOUND,
        wind_mps=[wind_spd * np.cos(wind_dir), wind_spd * np.sin(wind_dir), 0.0],
        weather=str(rng.choice(["clear", "overcast", "light_rain", "windy"])),
        bg_noise="airport_ambient.wav")

    sensors = [SensorSpec(**s) for s in C.SENSORS]
    return Scenario(
        scenario_id=f"dca_{seed:06d}",
        seed=int(seed),
        duration_s=float(duration_s),
        frame={"convention": "ENU", "origin": "tower_base_ground",
               "origin_latlon": [38.8523, -77.0378], "units": "m", "ground_z": 0},
        region_polygon=[list(p) for p in C.REGION_POLYGON],
        alt_band=[C.ALT_MIN, C.ALT_MAX],
        environment=env, objects=objects, sensors=sensors)


In [ ]:
%%writefile compile_track.py
"""
Compile a Layer-1 Scenario into a Layer-2 Track.

For each object:
  1. expand its maneuvers into positions sampled at MASTER_FS
  2. place them on the master timeline at start_delay (NaN where inactive)
  3. Gaussian-smooth the path so corners are rounded and velocity is well-defined
  4. derive velocity (np.gradient) and heading (yaw/pitch from velocity)
For each (object, sensor) pair, compute the physics-derived labels:
  range, radial velocity, az/el vs the sensor boresight, geometric in-FOV,
  expected received SPL (audio) and expected received power dBm (RF).

These labels are FEATURES derived from the world-state truth - not capture
thresholds. Audio/RF detectability emerges downstream from SNR; only the
camera has a hard geometric gate (and its occlusion test is added by the
visual renderer, which is the only stage that owns the mesh).
"""

from __future__ import annotations
import numpy as np
from schema import Scenario, Track
from maneuvers import expand
from geometry import smooth_columns
import config as C


def _fspl_db(d_m, f_hz):
    d = np.maximum(d_m, 1.0)
    return 20 * np.log10(d) + 20 * np.log10(f_hz) - 147.55


def _profile_for(obj):
    if obj.cls == "drone": return C.DRONE_PROFILES[obj.profile]
    if obj.cls == "bird":  return C.BIRD_PROFILES[obj.profile]
    return C.PLANE_PROFILES[obj.profile]


def _heading_from_vel(vel):
    vx, vy, vz = vel[:, 0], vel[:, 1], vel[:, 2]
    sp_h = np.hypot(vx, vy)
    yaw = np.degrees(np.arctan2(vy, vx))
    pitch = np.degrees(np.arctan2(vz, sp_h + 1e-9))
    # where nearly stationary, hold the previous heading (forward-fill)
    still = (np.hypot(sp_h, np.abs(vz)) < 0.2)
    for i in range(1, len(yaw)):
        if still[i]:
            yaw[i] = yaw[i - 1]; pitch[i] = pitch[i - 1]
    return yaw, pitch


def compile_scenario(scn: Scenario, fs=C.MASTER_FS) -> Track:
    rng = np.random.default_rng(scn.seed + 777)
    t = np.arange(0.0, scn.duration_s, 1.0 / fs)
    N = len(t)
    arrays: dict[str, np.ndarray] = {"t": t}
    threat = np.zeros(N, dtype=bool)

    sensors = scn.sensors
    sensor_pos = {s.id: np.asarray(s.pos, float) for s in sensors}
    # unit boresight vector per sensor from yaw/pitch
    boresight = {}
    for s in sensors:
        ya, pi = np.radians(s.yaw_deg), np.radians(s.pitch_deg)
        boresight[s.id] = np.array([np.cos(pi) * np.cos(ya),
                                    np.cos(pi) * np.sin(ya),
                                    np.sin(pi)])

    for obj in scn.objects:
        prof = _profile_for(obj)
        local_pos, dur = expand(obj.maneuvers, fs, obj.speed_mult, rng)
        local_t = np.arange(len(local_pos)) / fs + obj.start_delay
        active = (t >= obj.start_delay) & (t <= obj.start_delay + dur)

        pos = np.full((N, 3), np.nan)
        for j in range(3):
            pos[active, j] = np.interp(t[active], local_t, local_pos[:, j])
        # smooth only the active window (~150 ms)
        if active.sum() >= 3:
            pos[active] = smooth_columns(pos[active], sigma=0.15 * fs)

        vel = np.full((N, 3), np.nan)
        if active.sum() >= 2:
            vel[active] = np.gradient(pos[active], 1.0 / fs, axis=0)
        yaw = np.full(N, np.nan); pitch = np.full(N, np.nan)
        if active.sum() >= 2:
            yaw[active], pitch[active] = _heading_from_vel(vel[active])

        arrays[f"obj/{obj.id}/active"] = active
        arrays[f"obj/{obj.id}/pos"] = pos
        arrays[f"obj/{obj.id}/vel"] = vel
        arrays[f"obj/{obj.id}/yaw_deg"] = yaw
        arrays[f"obj/{obj.id}/pitch_deg"] = pitch
        if obj.cls == "drone":
            threat |= active

        # ----- per-sensor physics labels -----
        for s in sensors:
            sp = sensor_pos[s.id]
            los = pos - sp[None, :]                       # sensor -> object
            rng_m = np.linalg.norm(los, axis=1)
            unit = los / (rng_m[:, None] + 1e-9)
            radial = np.einsum("ij,ij->i", vel, unit)     # +ve = receding
            # az/el of LOS relative to the sensor boresight
            cosang = np.clip(los @ boresight[s.id] / (rng_m + 1e-9), -1, 1)
            off_axis = np.degrees(np.arccos(cosang))
            in_fov = off_axis <= (s.fov_deg / 2.0)
            # az in the horizontal plane relative to boresight yaw
            az = np.degrees(np.arctan2(los[:, 1], los[:, 0])) - s.yaw_deg
            az = (az + 180) % 360 - 180
            el = np.degrees(np.arctan2(los[:, 2], np.hypot(los[:, 0], los[:, 1]))) - s.pitch_deg
            spl = prof["ref_spl_db"] - 20 * np.log10(np.maximum(rng_m, 1.0))     # 1/r
            if obj.cls == "drone":
                rx = prof["rf_tx_dbm"] - _fspl_db(rng_m, prof["rf_carrier_hz"])
            else:
                rx = np.full(N, np.nan)                   # birds/planes emit no RF

            pre = f"lbl/{obj.id}/{s.id}"
            arrays[f"{pre}/range_m"] = rng_m
            arrays[f"{pre}/radial_mps"] = radial
            arrays[f"{pre}/az_deg"] = az
            arrays[f"{pre}/el_deg"] = el
            arrays[f"{pre}/in_fov"] = in_fov & active
            arrays[f"{pre}/spl_db"] = spl
            arrays[f"{pre}/rx_dbm"] = rx

    arrays["threat_present"] = threat

    meta = {
        "scenario_id": scn.scenario_id,
        "fs": fs,
        "duration_s": scn.duration_s,
        "is_threat": bool(threat.any()),
        "objects": [{"id": o.id, "cls": o.cls, "profile": o.profile} for o in scn.objects],
        "sensors": [{"id": s.id, "pos": s.pos, "yaw_deg": s.yaw_deg,
                     "pitch_deg": s.pitch_deg, "fov_deg": s.fov_deg,
                     "modalities": s.modalities} for s in sensors],
        "label_fields": ["range_m", "radial_mps", "az_deg", "el_deg",
                         "in_fov", "spl_db", "rx_dbm"],
    }
    return Track(arrays, meta)


## 2. Generate the dataset
Writes to `/kaggle/working/data` on Kaggle (persists with the notebook version).

In [ ]:
import os, json, importlib
import generator, compile_track, schema
importlib.reload(generator); importlib.reload(compile_track)
from generator import random_scenario
from compile_track import compile_scenario

# ---- knobs ----
N_SCENARIOS = 50
DURATION    = 45.0
N_DRONES, N_BIRDS, N_PLANES = 1, 1, 1   # set N_DRONES=0 for negative (no-threat) samples
SEED0       = 0
OUT = '/kaggle/working/data' if os.path.isdir('/kaggle/working') else 'data'
os.makedirs(OUT, exist_ok=True)

index = []
for i in range(N_SCENARIOS):
    seed = SEED0 + i
    scn = random_scenario(seed, DURATION, N_DRONES, N_BIRDS, N_PLANES)
    scn.to_json(os.path.join(OUT, f'{scn.scenario_id}.json'))
    track = compile_scenario(scn)
    track.save(os.path.join(OUT, f'track_{scn.scenario_id}.npz'),
               os.path.join(OUT, f'track_{scn.scenario_id}.meta.json'))
    index.append({'scenario_id': scn.scenario_id, 'is_threat': track.meta['is_threat'],
                  'n_objects': len(scn.objects)})
    if (i+1) % 10 == 0: print(f'{i+1}/{N_SCENARIOS} done')
json.dump(index, open(os.path.join(OUT,'index.json'),'w'), indent=2)
print('wrote', N_SCENARIOS, 'scenarios to', OUT)


## 3. Sanity check one track

In [ ]:
import numpy as np, glob
from schema import Track
mp = sorted(glob.glob(os.path.join(OUT,'track_*.meta.json')))[0]
tr = Track.load(mp.replace('.meta.json','.npz'), mp)
print('scenario', tr.meta['scenario_id'], '| objects', [(o['id'],o['cls']) for o in tr.meta['objects']])
oid = tr.object_ids[0]; sid = tr.meta['sensors'][0]['id']; act = tr.active(oid)
pos = tr.pos(oid)
print(f'{oid}: active {act.sum()}/{len(act)} samples')
print('  alt range %.0f..%.0f m' % (np.nanmin(pos[act,2]), np.nanmax(pos[act,2])))
print('  speed mean %.1f m/s' % np.nanmean(np.linalg.norm(tr.vel(oid)[act],axis=1)))
print(f'  range to {sid}: %.0f..%.0f m' % (np.nanmin(tr.label(oid,sid,'range_m')[act]), np.nanmax(tr.label(oid,sid,'range_m')[act])))
print('  npz arrays:', len(tr.a))


## 4. Save as a Kaggle Dataset (so other notebooks can use it)

Two easy ways:

**A. From notebook output (simplest):** click **Save Version** (top right) → after it commits, open the version, go to the **Output** tab → **New Dataset** → it packages everything in `/kaggle/working`.

**B. Zip first, then upload:** run the cell below to make a single zip in `/kaggle/working`, then Save Version and create a dataset from the output, or download the zip and upload via *Datasets → New Dataset*.


In [ ]:
import shutil, os
if os.path.isdir(OUT):
    shutil.make_archive('/kaggle/working/dca_groundtruth', 'zip', OUT)
    print('zipped ->', '/kaggle/working/dca_groundtruth.zip')
